In [1]:
import os
import json
import zipfile
import requests
import pandas as pd


RESPONDENT = "US48"
RESPONDENT_NAME = "United States Lower 48"

ZIP_PATH = "EBA.zip"
EXTRACT_DIR = "eia_bulk"
OUTPUT_DIR = "eia_yearly_csv"

BULK_URL = "https://www.eia.gov/opendata/bulk/EBA.zip"

# Only demand
KEEP_TYPES = {"D"}

TYPE_NAME_MAP = {
    "D": "Demand"
}



YEAR_RANGES = [
    ("2019-01-01T00", "2019-12-31T00", "2019"),
    ("2020-01-01T00", "2020-12-31T00", "2020"),
    ("2021-01-01T00", "2021-12-31T00", "2021"),
    ("2022-01-01T00", "2022-12-31T00", "2022"),
    ("2023-01-01T00", "2023-12-31T00", "2023"),
    ("2024-01-01T00", "2024-12-31T00", "2024"),
    ("2025-01-01T00", "2025-12-31T00", "2025"),
    ("2026-01-01T00", "2026-04-13T00", "2026_partial"),
]


def download_bulk_zip(url: str, out_path: str):
    if os.path.exists(out_path):
        print(f"ZIP already exists: {out_path}")
        return

    print(f"Downloading bulk file from:\n{url}")
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()

    total = 0
    with open(out_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                total += len(chunk)
                print(f"Downloaded {total / (1024 * 1024):.1f} MB", end="\r")

    print(f"\nSaved ZIP: {out_path}")


def extract_zip(zip_path: str, extract_dir: str) -> str:
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
        names = zf.namelist()

    txt_files = [n for n in names if n.lower().endswith(".txt")]
    if not txt_files:
        raise FileNotFoundError("No .txt file found inside the ZIP.")

    txt_path = os.path.join(extract_dir, txt_files[0])
    print(f"Extracted text file: {txt_path}")
    return txt_path


def normalize_period(period_str):
    if period_str is None:
        return pd.NaT

    s = str(period_str).strip()
    dt = pd.to_datetime(s, errors="coerce", utc=True)

    if pd.isna(dt):
        return pd.NaT

    return dt.tz_convert(None)


def format_period_for_output(ts):
    if pd.isna(ts):
        return None
    return ts.strftime("%Y-%m-%dT%H")


def detect_type(series_id: str):
    if not isinstance(series_id, str):
        return None

    sid = series_id.upper()

    if ".DF." in sid or sid.endswith(".DF"):
        return "DF"
    if ".D." in sid or sid.endswith(".D"):
        return "D"
    if ".NG." in sid or sid.endswith(".NG"):
        return "NG"
    if ".TI." in sid or sid.endswith(".TI"):
        return "TI"

    return None


def is_us48_series(obj):
    sid = str(obj.get("series_id", ""))
    name = str(obj.get("name", ""))
    text = f"{sid} {name}".upper()
    return RESPONDENT in text and "data" in obj


def collect_all_demand_rows(txt_path: str):
    rows = []
    matched_series = 0
    total_lines = 0

    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            total_lines += 1
            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            if "series_id" not in obj or "data" not in obj:
                continue

            if not is_us48_series(obj):
                continue

            series_id = obj.get("series_id", "")
            data = obj.get("data", [])
            units = obj.get("units", "megawatthours")

            typ = detect_type(series_id)

            if typ not in KEEP_TYPES:
                continue

            matched_series += 1

            for item in data:
                if not isinstance(item, list) or len(item) < 2:
                    continue

                raw_period = item[0]
                value = item[1]

                dt = normalize_period(raw_period)
                if pd.isna(dt):
                    continue

                rows.append({
                    "period_dt": dt,
                    "period": format_period_for_output(dt),
                    "respondent": RESPONDENT,
                    "respondent-name": RESPONDENT_NAME,
                    "type": typ,
                    "type-name": TYPE_NAME_MAP.get(typ, typ),
                    "value": pd.to_numeric(value, errors="coerce"),
                    "value-units": units if units else "megawatthours"
                })

            if matched_series % 10 == 0:
                print(
                    f"Scanned lines: {total_lines:,} | "
                    f"matched series: {matched_series:,} | "
                    f"rows collected: {len(rows):,}"
                )

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError("No demand rows found in the bulk file.")

    return df


def save_yearly_csvs(df_all: pd.DataFrame, year_ranges, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)

    saved_files = []

    for start, end, label in year_ranges:
        start_dt = pd.to_datetime(start, utc=True).tz_convert(None)
        end_dt = pd.to_datetime(end, utc=True).tz_convert(None)

        df_year = df_all[
            (df_all["period_dt"] >= start_dt) &
            (df_all["period_dt"] <= end_dt)
        ].copy()

        df_year = df_year.sort_values("period_dt", ascending=False).reset_index(drop=True)

        # drop helper datetime column before saving
        df_year = df_year.drop(columns=["period_dt"])

        output_path = os.path.join(output_dir, f"eia_us48_demand_hourly_utc_{label}.csv")
        df_year.to_csv(output_path, index=False)

        print(f"Saved: {output_path} | Shape: {df_year.shape}")
        saved_files.append(output_path)

    return saved_files


def main():
    download_bulk_zip(BULK_URL, ZIP_PATH)
    txt_path = extract_zip(ZIP_PATH, EXTRACT_DIR)

    print("\nCollecting all US48 demand rows once...")
    df_all = collect_all_demand_rows(txt_path)

    print(f"\nTotal collected demand rows: {df_all.shape[0]:,}")

    saved_files = save_yearly_csvs(df_all, YEAR_RANGES, OUTPUT_DIR)

    print("\nDone. Files created:")
    for path in saved_files:
        print(path)

    return df_all, saved_files


if __name__ == "__main__":
    df_all, saved_files = main()

https://www.eia.gov/opendata/bulk/EBA.zip
Downloaded 637.7 MB
Saved ZIP: EBA.zip
Extracted text file: eia_bulk/EBA.txt


Total collected demand rows: 127,554
Saved: eia_yearly_csv/eia_us48_demand_hourly_utc_2019.csv | Shape: (17474, 7)
Saved: eia_yearly_csv/eia_us48_demand_hourly_utc_2020.csv | Shape: (17522, 7)
Saved: eia_yearly_csv/eia_us48_demand_hourly_utc_2021.csv | Shape: (17474, 7)
Saved: eia_yearly_csv/eia_us48_demand_hourly_utc_2022.csv | Shape: (17474, 7)
Saved: eia_yearly_csv/eia_us48_demand_hourly_utc_2023.csv | Shape: (17474, 7)
Saved: eia_yearly_csv/eia_us48_demand_hourly_utc_2024.csv | Shape: (17522, 7)
Saved: eia_yearly_csv/eia_us48_demand_hourly_utc_2025.csv | Shape: (17474, 7)
Saved: eia_yearly_csv/eia_us48_demand_hourly_utc_2026_partial.csv | Shape: (4818, 7)

Done. Files created:
eia_yearly_csv/eia_us48_demand_hourly_utc_2019.csv
eia_yearly_csv/eia_us48_demand_hourly_utc_2020.csv
eia_yearly_csv/eia_us48_demand_hourly_utc_2021.csv
eia_yearly_csv/eia_us48_demand_hourl